In [1]:
import os
import gc
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", None)

# NOTE: Avoid importing packages that can crash in this environment (e.g., nltk/textblob/protobuf-related).
# Use tf.keras to ensure all Keras APIs exist consistently on Kaggle.
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split

seed = 101
np.random.seed(seed)
tf.random.set_seed(seed)

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))



2026-05-12 05:35:19.157516: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 05:35:19.183887: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778564119.212410      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778564119.220998      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 05:35:19.253035: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/kaggle/input/train.tsv.zip
/kaggle/input/sampleSubmission.csv
/kaggle/input/test.tsv.zip
/kaggle/input/description.md
/kaggle/input/sampleSubmission.csv.zip
/kaggle/input/test.tsv
/kaggle/input/train.tsv
/kaggle/input/movie-review-sentiment-analysis-kernels-only/train.tsv.zip
/kaggle/input/movie-review-sentiment-analysis-kernels-only/sampleSubmission.csv
/kaggle/input/movie-review-sentiment-analysis-kernels-only/test.tsv.zip
/kaggle/input/movie-review-sentiment-analysis-kernels-only/description.md
/kaggle/input/movie-review-sentiment-analysis-kernels-only/sampleSubmission.csv.zip
/kaggle/input/movie-review-sentiment-analysis-kernels-only/test.tsv
/kaggle/input/movie-review-sentiment-analysis-kernels-only/train.tsv


In [2]:
# Robustly resolve paths (Kaggle typically uses /kaggle/input/...)
BASE = "/kaggle/input/movie-review-sentiment-analysis-kernels-only"
if not os.path.exists(BASE):
    BASE = "/kaggle/input"

train_path = os.path.join(BASE, "train.tsv.zip")
test_path = os.path.join(BASE, "test.tsv.zip")
sub_path = os.path.join(BASE, "sampleSubmission.csv")

train = pd.read_csv(train_path, sep="\t", encoding="utf-8")
test = pd.read_csv(test_path, sep="\t", encoding="utf-8")
sub = pd.read_csv(sub_path)

print(train.shape)
print(test.shape)
print(sub.shape)



(109242, 4)
(46818, 3)
(46818, 2)


In [3]:
train.head()



,PhraseId,SentenceId,Phrase,Sentiment
0,24144,1097,through every frame,2
1,85111,4402,onscreen presence,2
2,75999,3900,into dysfunction,2
3,17377,754,a probe,2
4,9664,403,half hour,2


In [4]:
test.head()



,PhraseId,SentenceId,Phrase
0,21924,980,escape movie
1,18207,795,well-executed
2,81803,4221,flick formula
3,114983,6124,is too savvy a filmmaker to let this morph into a typical romantic triangle .
4,135928,7343,"There are laughs aplenty ,"


In [5]:
# Prepare features/labels
X = train["Phrase"].astype(str)
temp = test["Phrase"].astype(str)

y_int = train["Sentiment"].astype(int).values  # for stratify/splitting
num_classes = train["Sentiment"].nunique()
print("Number of classes:", num_classes)

y = to_categorical(y_int, num_classes=num_classes)  # one-hot for training



Number of classes: 5


In [6]:
# Split Train/Validation (stratify must be 1D class labels, not one-hot)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y_int, random_state=seed
)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)



(87393,) (21849,) (87393, 5) (21849, 5)


In [7]:
# Tokenization + padding (same core approach)
max_features = 15000
tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(list(X_train))

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
temp_seq = tokenizer.texts_to_sequences(temp)

max_words = 50
X_train_seq = sequence.pad_sequences(X_train_seq, maxlen=max_words)
X_val_seq = sequence.pad_sequences(X_val_seq, maxlen=max_words)
temp_seq = sequence.pad_sequences(temp_seq, maxlen=max_words)

print(X_train_seq.shape, X_val_seq.shape, temp_seq.shape)



(87393, 50) (21849, 50) (46818, 50)


In [8]:
# Embedding matrix:
# Original code expected external GloVe file, but it's not available in provided inputs.
# Minimal fix: keep same Embedding+LSTM architecture, but use random init embedding_matrix with correct shape.
embed_dim = 100
word_index = tokenizer.word_index
num_words = min(max_features, len(word_index) + 1)

rng = np.random.default_rng(seed)
embedding_matrix = rng.normal(loc=0.0, scale=0.05, size=(num_words, embed_dim)).astype(
    np.float32
)
max_features_eff = embedding_matrix.shape[0]

print("Vocab size used:", max_features_eff)
print("Embedding matrix:", embedding_matrix.shape)



Vocab size used: 15000
Embedding matrix: (15000, 100)


In [9]:
batch_size = 128
epochs = 10


def get_model(max_features, embed_dim, embedding_matrix):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    K.clear_session()

    model = Sequential()
    model.add(
        Embedding(
            input_dim=max_features,
            output_dim=embed_dim,
            input_length=X_train_seq.shape[1],
            weights=[embedding_matrix],
        )
    )
    model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
    model.add(Dense(num_classes, activation="softmax"))
    model.compile(
        loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
    )
    model.summary()
    return model


model = get_model(max_features_eff, embed_dim, embedding_matrix)
model.fit(
    X_train_seq,
    y_train,
    validation_data=(X_val_seq, y_val),
    epochs=epochs,
    batch_size=batch_size,
    verbose=2,
)



2026-05-12 05:35:37.269171: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     1,500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,500,000 (5.72 MB)

 Trainable params: 1,500,000 (5.72 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
683/683 - 48s - 71ms/step - accuracy: 0.5804 - loss: 1.0415 - val_accuracy: 0.6339 - val_loss: 0.8916
Epoch 2/10
683/683 - 40s - 58ms/step - accuracy: 0.6708 - loss: 0.8021 - val_accuracy: 0.6527 - val_loss: 0.8570
Epoch 3/10
683/683 - 36s - 53ms/step - accuracy: 0.7023 - loss: 0.7212 - val_accuracy: 0.6563 - val_loss: 0.8624
Epoch 4/10
683/683 - 31s - 46ms/step - accuracy: 0.7230 - loss: 0.6649 - val_accuracy: 0.6563 - val_loss: 0.8865
Epoch 5/10
683/683 - 31s - 45ms/step - accuracy: 0.7393 - loss: 0.6202 - val_accuracy: 0.6534 - val_loss: 0.9155
Epoch 6/10
683/683 - 41s - 60ms/step - accuracy: 0.7547 - loss: 0.5808 - val_accuracy: 0.6522 - val_loss: 0.9547
Epoch 7/10
683/683 - 54s - 80ms/step - accuracy: 0.7647 - loss: 0.5473 - val_accuracy: 0.6512 - val_loss: 1.0002
Epoch 8/10
683/683 - 61s - 90ms/step - accuracy: 0.7780 - loss: 0.5179 - val_accuracy: 0.6494 - val_loss: 1.0503
Epoch 9/10
683/683 - 58s - 86ms/step - accuracy: 0.7872 - loss: 0.4937 - val_accuracy: 0.6463 - 

In [10]:
# Submission:
# predict_classes is removed in modern tf.keras; use argmax over softmax probabilities.
probs = model.predict(temp_seq, batch_size=batch_size, verbose=0)
pred = np.argmax(probs, axis=1).astype(int)

sub["Sentiment"] = pred
sub = sub[["PhraseId", "Sentiment"]]
sub.to_csv("submission.csv", index=False)
sub.head()



,PhraseId,Sentiment
0,21924,2
1,18207,4
2,81803,2
3,114983,1
4,135928,3


In [11]:
sub.head()

,PhraseId,Sentiment
0,21924,2
1,18207,4
2,81803,2
3,114983,1
4,135928,3
